In [3]:
import pandas as pd
import numpy as np

SRC = "calendar_enriched.csv"
OUT = "calendar_clean.csv"

df = pd.read_csv(SRC, parse_dates=["date"])

df = df.sort_values(["park_id", "date"]).reset_index(drop=True)
flag_cols = ["is_weekend", "has_early_entry", "has_extended_evening",
             "has_special_event", "is_holiday", "is_rainy"]
df[flag_cols] = df[flag_cols].astype("int8")
df["park_name"] = df["park_name"].astype("category")

df["op_minutes_placeholder"] = (df["operating_minutes"] == 1440).astype("int8")
df.loc[df["op_minutes_placeholder"] == 1, "operating_minutes"] = np.nan

df["op_minutes_imputed"] = df["operating_minutes"].isna().astype("int8")
df["operating_minutes_filled"] = (
    df.groupby(["park_id", "year", "dow_number"])["operating_minutes"]
      .transform(lambda s: s.fillna(s.median()))
)
df["operating_minutes_filled"] = df["operating_minutes_filled"].fillna(
    df.groupby(["park_id", "dow_number"])["operating_minutes"].transform("median")
)

df["crowd_zero_flag"] = (df["crowd_level_pct"] == 0).astype("int8")

df["short_day_flag"] = (df["operating_minutes_filled"] < 300).astype("int8")

df = df.drop(columns=["month_name", "day_of_week"])


assert not df.duplicated(["park_id", "date"]).any()
assert df["crowd_level_pct"].between(0, 100).all()
assert df["operating_minutes_filled"].notna().all()

df.to_csv(OUT, index=False)
print(df.shape)
print(df.filter(like="flag").sum())
print(df.filter(like="op_minutes").sum())

(16682, 20)
crowd_zero_flag    86
short_day_flag      1
dtype: int64
op_minutes_placeholder     11
op_minutes_imputed        641
dtype: int64
